In [1]:
import os

In [2]:
%pwd

'c:\\Users\\pm062\\Desktop\\End_to_End_TEXT_SUMMARIZER\\research'

In [3]:
os.chdir("../")

In [4]:
from dataclasses import dataclass
from pathlib import Path

@dataclass(frozen=True)
class DataIngestionConfig:
    root_dir: Path
    source_URL: str
    local_data_file: Path
    unzip_dir: Path

In [5]:
from text_summarizer.constants import *
from text_summarizer.utils.common import read_yaml, create_directories

In [6]:
class ConfigurationManager:
    def __init__(
            self,
            config_filepath = CONFIG_FILE_PATH,
            params_filepath = PARAMS_FILE_PATH):
        
            self.config = read_yaml(config_filepath)
            self.params = read_yaml(params_filepath)

            create_directories([self.config.artifacts_root])

    def get_data_ingestion_config(self) -> DataIngestionConfig:
          config = self.config.data_ingestion

          create_directories([config.root_dir])

          data_ingestion_config = DataIngestionConfig(
            root_dir=config.root_dir,
            source_URL=config.source_URL,
            local_data_file=config.local_data_file,
            unzip_dir=config.unzip_dir
            )
          return data_ingestion_config

In [7]:
import os
import urllib
import urllib.request as request
import zipfile
from text_summarizer.logging import logger
from text_summarizer.utils.common import get_size

In [8]:
class DataIngestion:
    def __init__(self, config: DataIngestionConfig):
        self.config = config

    def download_file(self):
        if not os.path.exists(self.config.local_data_file):
            filename, headers = request.urlretrieve(
                url = self.config.source_URL,
                filename= self.config.local_data_file
            )
            logger.info(f"{filename} download? with following info: \n{headers}")
        else:
            logger.info(f"File already exists of size: {get_size(Path(self.config.local_data_file))}")

    def extract_zip_file(self):
        """
        Extracts the zip file into the data directory.
        """
        unzip_path = self.config.unzip_dir

        os.makedirs(unzip_path, exist_ok=True)

        if not os.path.exists(self.config.local_data_file):
            raise FileNotFoundError(
                f"Zip file not found: {self.config.local_data_file}"
            )

        with zipfile.ZipFile(self.config.local_data_file, "r") as zip_ref:
            zip_ref.extractall(unzip_path)

        logger.info(f"Files extracted to {unzip_path}")

In [9]:
try:
    config = ConfigurationManager()
    data_ingestion_config = config.get_data_ingestion_config()
    data_ingestion = DataIngestion(config=data_ingestion_config)
    data_ingestion.download_file()
    data_ingestion.extract_zip_file()
except Exception as e:
    raise e

[2026-07-10 17:09:19,047: INFO: common: yaml file: config\config.yaml loaded successfully]
[2026-07-10 17:09:19,050: INFO: common: yaml file: params.yaml loaded successfully]
[2026-07-10 17:09:19,050: INFO: common: created directory at:artifacts]
[2026-07-10 17:09:19,057: INFO: common: created directory at:artifacts/data_ingestion]
[2026-07-10 17:09:20,586: INFO: 827619634: artifacts/data_ingestion/data.zip download? with following info: 
Connection: close
Content-Length: 7903594
Cache-Control: max-age=300
Content-Security-Policy: default-src 'none'; style-src 'unsafe-inline'; sandbox
Content-Type: application/zip
ETag: "dbc016a060da18070593b83afff580c9b300f0b6ea4147a7988433e04df246ca"
Strict-Transport-Security: max-age=31536000
X-Content-Type-Options: nosniff
X-Frame-Options: deny
X-XSS-Protection: 1; mode=block
X-GitHub-Request-Id: F228:25F31F:47D32A:8A51F5:6A50D9E7
Accept-Ranges: bytes
Date: Fri, 10 Jul 2026 11:39:20 GMT
Via: 1.1 varnish
X-Served-By: cache-maa10236-MAA
X-Cache: MISS

In [10]:
print(data_ingestion_config.source_URL)

https://raw.githubusercontent.com/PrathiPanthera/End_to_End_TEXT_SUMMARIZER/main/summarizer-data.zip
